# Weekday Ridership and Stop Data Cleaning, Standardization, and Integration Pipeline

This script prepares and integrates transit ridership data with stop-level data by cleaning inconsistencies, applying agency-specific fixes, and merging datasets using appropriate matching strategies.

**Summary**
- Load & filter data: Ridership data is standardized and filtered to weekdays; stop data is aligned with consistent agency names.
- Clean data: Fixes inconsistencies in route_id, stop_id, and stop_code across agencies to improve matching.
- Split by strategy: Agencies are grouped based on how they can be matched (route + stop, stop_code, or stop_name).
- Merge datasets: Each group is merged with stop data, keeping only successful matches.
- Combine results: All matched data is concatenated into a final dataset linking weekday ridership with stops and routes.

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()
from rapidfuzz import process, fuzz

pd.set_option('display.max_columns', None)

In [3]:
# Read in ridership data 
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [5]:
ridership_data_weekday = (
    pd.read_csv(f"{GCS_FILE_PATH}/ridership_all.csv")
    .assign(
        route_id=lambda df: df['route_id'].apply(
            lambda x: str(int(x)) if isinstance(x, float) and not pd.isna(x) else str(x)
        )
    )
    .sort_values(by='stop_name')
    .loc[lambda df: df['day_type'] == "Weekday"]
    .reset_index(drop=True)
)

ridership_data_weekday.info()
ridership_data_weekday.head(5)

/tmp/ipykernel_1204/444638858.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(f"{GCS_FILE_PATH}/ridership_all.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30707 entries, 0 to 30706
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   organization_name         30707 non-null  object 
 1   route_id                  30707 non-null  object 
 2   route_name                14875 non-null  object 
 3   stop_id                   30677 non-null  object 
 4   stop_name                 25210 non-null  object 
 5   stop_lat                  9787 non-null   float64
 6   stop_lon                  9787 non-null   float64
 7   day_type                  30707 non-null  object 
 8   average_daily_boardings   30707 non-null  float64
 9   average_daily_alightings  27592 non-null  float64
 10  start_date                30707 non-null  object 
 11  end_date                  30707 non-null  object 
dtypes: float64(4), object(8)
memory usage: 2.8+ MB


,organization_name,route_id,route_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
0,Gold Coast Transit,16,NaN,VNACLR1,.,34.343645,-119.294028,Weekday,0.000000,3.000000,2025-05-01,2025-05-31
1,Samtrans,ECR,NaN,345017,1000 El Camino Real-Menlo College,37.457839,-122.190513,Weekday,9.523810,24.571429,2025-08-01,2025-08-31
2,Golden Gate Transit,130,NaN,40469,1011 Andersen Dr,NaN,NaN,Weekday,1.909091,0.000000,2025-09-01,2025-09-30
3,Samtrans,61,NaN,343119,1011 Crestview Dr,37.484237,-122.284042,Weekday,1.444444,1.888889,2025-08-01,2025-08-31
4,Samtrans,46,NaN,340606,1060 Carolan Ave,37.587028,-122.359191,Weekday,10.333333,5.500000,2025-08-01,2025-08-31


In [6]:
agency_mapping = {
    'Gold Coast Schedule': 'Gold Coast Transit',
    'SamTrans Schedule': 'Samtrans',
    'Big Blue Bus Schedule': 'Big Blue Bus',
    'Foothill Schedule': 'Foothill Transit',
    'San Diego Schedule': 'SDMTS',
    'Golden Gate Park Shuttle Schedule': 'Golden Gate Park Shuttle',
    'Fresno Schedule': 'Fresno County',
    'OmniTrans Schedule': 'Omnitrans',
    'Sacramento Schedule': 'SacRT Bus',
    'Bay Area 511 BART Schedule': 'San Francisco Bay Area Rapid Transit District',
    'Riverside Schedule': 'Riverside Transit',
    'OCTA Schedule': 'Orange County Transportation Authority',
    'Santa Cruz Schedule': 'Santa Cruz Metro',
    'Long Beach Schedule': 'Long Beach Transit',
    'Caltrain Schedule': 'Caltrain',
    'SBMTD Schedule': 'SBMTD',
    'Culver City Schedule': 'Culver City Bus',
    'Bay Area 511 Golden Gate Transit Schedule': 'Golden Gate Transit'
}

### Weekday Stops Data Processing

In [7]:
stops_aggregated_weekday = (
    pd.read_csv(f"{GCS_FILE_PATH}/stops_aggregated_weekday.csv")
    .assign(organization_name=lambda df: df['name'].map(agency_mapping))
)

stops_aggregated_weekday.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36383 entries, 0 to 36382
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   feed_key           36383 non-null  object 
 1   stop_id            36383 non-null  object 
 2   stop_name          36383 non-null  object 
 3   route_id           33395 non-null  object 
 4   route_type         33395 non-null  float64
 5   n_trips            36383 non-null  int64  
 6   n_routes           36383 non-null  int64  
 7   gtfs_dataset_key   36383 non-null  object 
 8   name               36383 non-null  object 
 9   route_long_name    32361 non-null  object 
 10  stop_code          32888 non-null  object 
 11  pt_geom            36383 non-null  object 
 12  organization_name  36383 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.6+ MB


In [8]:
# Upon Checking found that Samtrans Transit Route Id has -216 in the stops aggregated data, while ridership data we have doesnot. So removing -216 from stopsa_aggregated data 
stops_aggregated_weekday.loc[
    stops_aggregated_weekday['name'] == "SamTrans Schedule",
    'route_id'
] = stops_aggregated_weekday.loc[
    stops_aggregated_weekday['name'] == "SamTrans Schedule",
    'route_id'
].str.replace('-216$', '', regex=True)


# Upon checking, found that Gold Coast Schedule stop_ids have some issues:
# 1. Some stop_ids contain ':' which prevents correct merge with ridership data
# 2. Two specific stop_ids ('GCTDMAIN1' and 'SAClMAI1') do not match the ridership data stop_ids ('GCTDMAIN' and 'SACLMAI1')
# So we remove colons and update these two stop_ids before merging
stops_aggregated_weekday.loc[
    stops_aggregated_weekday['name'] == "Gold Coast Schedule",
    'stop_id'
] = stops_aggregated_weekday.loc[
    stops_aggregated_weekday['name'] == "Gold Coast Schedule",
    'stop_id'
].str.replace(':', '', regex=False).replace({
    'GCTDMAIN1': 'GCTDMAIN',
    'SAClMAI1': 'SACLMAI1'
})



# Upon checking, found that Some stops are unmatched because Big Blue Bus ridership uses negative stop_codes.
# These correspond to valid stop_codes in the stop data (same route and stop_name).
# The mapping below replaces negative ridership stop_codes with the correct stop_codes
# so they can match during the merge.

# Apply only for Big Blue Bus
mask_bb = ridership_data_weekday['organization_name'] == 'Big Blue Bus'

# Convert to int first
ridership_data_weekday.loc[mask_bb, 'stop_id'] = (
    ridership_data_weekday.loc[mask_bb, 'stop_id']
    .astype(int)
    .replace({
        -4: 'MNSWSMNF',
        -5: '014MCHNN',
        -6: '014PCONF',
        -7: '014PCOSF',
        -8: '017PRLNN',
        -9: '017PRLSF',
        -13: 'PRL014EF',
        -14: 'PRL014WN',
        -18: 'SMCBUNPL'
    })
    .astype(str)
)


stops_aggregated_weekday.loc[
    stops_aggregated_weekday['organization_name'] == 'Big Blue Bus',
    'stop_code'
] = stops_aggregated_weekday.loc[
    stops_aggregated_weekday['organization_name'] == 'Big Blue Bus',
    'stop_code'
].astype(str).str.strip()


# Upon checking found that some route_ids are differently named in ridership data compared to stop_data so changing that to match
# Filter for Culver City Bus
mask_cc = ridership_data_weekday['organization_name'] == 'Culver City Bus'

# Standardize route_id in ridership
ridership_data_weekday.loc[mask_cc, 'route_id'] = (
    ridership_data_weekday.loc[mask_cc, 'route_id']
    .replace({
        '1C1': '1c1',
        'Rapid 6': '6R'
    })
)

# Standardize route_id in stops
stops_aggregated_weekday.loc[
    stops_aggregated_weekday['organization_name'] == 'Culver City Bus',
    'route_id'
] = stops_aggregated_weekday.loc[
    stops_aggregated_weekday['organization_name'] == 'Culver City Bus',
    'route_id'
].replace({
    '1C1': '1c1',
    'Rapid 6': '6R'
})


# Upon checking found that Long Beach transit and OCTA stop_id has leading zeros like 0110, 0220 and so on
for agency in ['Long Beach Transit', 'Orange County Transportation Authority']:
    stops_aggregated_weekday.loc[
        stops_aggregated_weekday['organization_name'] == agency,
        'stop_id'
    ] = (
        stops_aggregated_weekday.loc[
            stops_aggregated_weekday['organization_name'] == agency,
            'stop_id'
        ]
        .str.replace(r'^0+', '', regex=True)
    )




In [9]:
master_cols = [
    "organization_name",
    "feed_key",
    "route_id",
    "route_name",
    "stop_id",
    "stop_name",
    "stop_code",
    "n_trips",
    "n_routes",
    "pt_geom",
    "day_type",
    "average_daily_boardings",
    "average_daily_alightings",
    "start_date",
    "end_date"
]

def standardize_columns(df, master_cols):
    
    # add missing columns
    for col in master_cols:
        if col not in df.columns:
            df[col] = None

    # keep only master columns and order them
    return df[master_cols]

### Agencies with Weekday Data, Categorized by Match Type: route*stop_id, stop_code, or stop_name

In [10]:
agencies_with_route = [
    'Foothill Transit',
    'Gold Coast Transit',
    'Golden Gate Transit',
    'Long Beach Transit',
    'Orange County Transportation Authority',
    # 'Omnitrans',    
    'SacRT Bus',
    'Samtrans',
    'SDMTS'
]

agencies_with_route_stop_code_match = [
    'Culver City Bus',
    'Big Blue Bus',
    'Riverside Transit'
]


agencies_without_route = ['San Francisco Bay Area Rapid Transit District', 
                          'Fresno County',
                          # 'SBMTD'
                         ]


agencies_without_route_stop_code_match = [
                          # 'Santa Cruz Metro',
                          'Golden Gate Park Shuttle',
                         ]


agencies_without_route_stop_name_match = [
                          'Caltrain'
                         ]

In [11]:
# Split Ridership by categories

def split_ridership_by_agency(ridership_df, agency_groups):
    subsets = {}
    for name, agencies in agency_groups.items():
        subsets[name] = ridership_df[ridership_df['organization_name'].isin(agencies)]
    return subsets

agency_groups = {
    'with_route': agencies_with_route,
    'without_route': agencies_without_route,
    'with_route_stopcode_match': agencies_with_route_stop_code_match,
    'without_route_stopcode_match': agencies_without_route_stop_code_match,
    'without_route_stopname_match': agencies_without_route_stop_name_match
}

ridership_subsets = split_ridership_by_agency(ridership_data_weekday, agency_groups)



In [12]:
# Merge with route_id
merge_with_route = (
    pd.merge(
        ridership_subsets['with_route'],  
        stops_aggregated_weekday,
        on=['organization_name', 'route_id', 'stop_id'],
        how='left',
        indicator=True,
        suffixes=('_x','_y')
    )
    .rename(columns={'stop_name_y': 'stop_name'})  # rename matched stop_name
)

# Keep only rows where the merge matched in both DataFrames
merge_with_route_matched = merge_with_route[merge_with_route['_merge'] == 'both'].copy()


# Optional: inspect merge results
merge_counts = merge_with_route['_merge'].value_counts()
print(merge_counts)

_merge
both          22802
left_only      1331
right_only        0
Name: count, dtype: int64


In [13]:
merge_with_route_matched = standardize_columns(merge_with_route_matched, master_cols)

In [14]:
# Merge without route_id
merge_without_route = (
    pd.merge(
        ridership_subsets['without_route'], 
        stops_aggregated_weekday,
        on=['organization_name', 'stop_id'],
        how='left',
        indicator=True,
        suffixes=('_x','_y')
    )
    .rename(columns={
        'stop_name_y': 'stop_name',
        'route_id_y': 'route_id'
    })
)

# Keep only rows where the merge matched in both DataFrames
merge_without_route_matched = merge_without_route[merge_without_route['_merge'] == 'both'].copy()

merge_counts = merge_without_route['_merge'].value_counts()
print(merge_counts)

_merge
both          1622
left_only       47
right_only       0
Name: count, dtype: int64


In [15]:
merge_without_route_matched = standardize_columns(merge_without_route_matched, master_cols)
merge_without_route_matched.sample(5)

,organization_name,feed_key,route_id,route_name,stop_id,stop_name,stop_code,n_trips,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
71,San Francisco Bay Area Rapid Transit District,4e549244dd7ce383a3f4337f00134b27,NaN,NaN,90950,Berryessa / North San Jose,909501,261.0,4.0,POINT(-121.874759 37.368474),Weekday,1447.413793,1497.176245,2024-10-01,2025-09-30
189,Fresno County,23d1893801eefadf7544a670a3bcd312,NaN,NaN,2258,NE ALLUVIAL - FRESNO,2258,118.0,1.0,POINT(-119.784248 36.84469),Weekday,2.463356,12.049631,2024-09-01,2025-08-31
1128,Fresno County,23d1893801eefadf7544a670a3bcd312,NaN,NaN,2343,SE MARKS - WELDEN,2343,64.0,1.0,POINT(-119.844144 36.768475),Weekday,4.654768,9.836201,2024-09-01,2025-08-31
704,Fresno County,23d1893801eefadf7544a670a3bcd312,NaN,NaN,273,NW GETTYSBURG - WILLOW,273,119.0,1.0,POINT(-119.727725 36.801259),Weekday,22.002434,15.939271,2024-09-01,2025-08-31
1275,Fresno County,23d1893801eefadf7544a670a3bcd312,NaN,NaN,1563,SE WALNUT - CALIFORNIA,1563,112.0,1.0,POINT(-119.808879 36.720821),Weekday,55.284446,23.804039,2024-09-01,2025-08-31


In [16]:
merge_with_route_stopcode = (
    pd.merge(
        ridership_subsets['with_route_stopcode_match'],  
        stops_aggregated_weekday,
        left_on=['organization_name', 'route_id', 'stop_id'],  # ridership stop_id
        right_on=['organization_name', 'route_id', 'stop_code'],  # stops table stop_code
        how='left',
        indicator=True
    )
    .rename(columns={
        'stop_id_y': 'stop_id',
        'stop_name_y': 'stop_name',
    })
)


# Keep only rows where the merge matched in both DataFrames
merge_with_route_stopcode_matched = merge_with_route_stopcode[merge_with_route_stopcode['_merge'] == 'both'].copy()

# Check merge results
merge_counts = merge_with_route_stopcode['_merge'].value_counts()
print(merge_counts)

_merge
both          4073
left_only      785
right_only       0
Name: count, dtype: int64


In [17]:
merge_with_route_stopcode_matched = standardize_columns(merge_with_route_stopcode_matched, master_cols)
merge_with_route_stopcode_matched.sample(5)

,organization_name,feed_key,route_id,route_name,stop_id,stop_name,stop_code,n_trips,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3090,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,15,NaN,606,La Sierra + Nebraska,3615,50.0,1.0,POINT(-117.489828 33.921166),Weekday,7.027027,NaN,2025-01-01,2025-10-31
2291,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,9,NaN,1237,Fourth + A Street,2386,29.0,1.0,POINT(-117.235197 33.782261),Weekday,10.195876,NaN,2025-01-01,2025-10-31
697,Big Blue Bus,7a3f513c343b16a30c135ed7d332b6d6,3,Lincoln Blvd/LAX,78,LINCOLN BLVD & PICO BLVD,1212,153.0,1.0,POINT(-118.48259 34.011813),Weekday,153.616562,31.137641,2024-08-01,2025-11-30
2512,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,11,NaN,857,Heacock + Dracaea,2047,20.0,1.0,POINT(-117.243871 33.927824),Weekday,2.437500,NaN,2025-01-01,2025-10-31
918,Big Blue Bus,7a3f513c343b16a30c135ed7d332b6d6,8,Ocean Park Blvd & Westwood Bl/UCLA,833,OCEAN PARK BLVD & 11TH ST,1411,71.0,1.0,POINT(-118.474068 34.008346),Weekday,2.395096,8.382044,2024-08-01,2025-11-30


In [18]:
merge_without_route_stopcode = ( 
    pd.merge(
    ridership_subsets['without_route_stopcode_match'], 
    stops_aggregated_weekday,
    left_on=['organization_name', 'stop_id'],   # ridership stop_id
    right_on=['organization_name', 'stop_code'],  # stops table stop_code
    how='left',
    indicator=True
    )
    .rename(columns={
        'stop_id_y': 'stop_id',
        'stop_name_y': 'stop_name',
        'route_id_y': 'route_id'
    })
)

# Keep only rows where the merge matched in both DataFrames
merge_without_route_stopcode_matched = merge_without_route_stopcode[merge_without_route_stopcode['_merge'] == 'both'].copy()


merge_counts = merge_without_route_stopcode['_merge'].value_counts()
print(merge_counts)

_merge
both          18
left_only      0
right_only     0
Name: count, dtype: int64


In [19]:
merge_without_route_stopcode_matched = standardize_columns(merge_without_route_stopcode_matched, master_cols)
merge_without_route_stopcode_matched.sample(5)

,organization_name,feed_key,route_id,route_name,stop_id,stop_name,stop_code,n_trips,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
17,Golden Gate Park Shuttle,ea33d4691b573336fc9c43c23fa90f65,NaN,NaN,TJFK,Transverse,7601,32,1,POINT(-122.479657 37.770534),Weekday,9.118774,NaN,2024-07-01,2025-06-30
9,Golden Gate Park Shuttle,ea33d4691b573336fc9c43c23fa90f65,NaN,NaN,HS,Haight / Stanyan,7618,32,1,POINT(-122.45295 37.76927),Weekday,10.379310,NaN,2024-07-01,2025-06-30
12,Golden Gate Park Shuttle,ea33d4691b573336fc9c43c23fa90f65,NaN,NaN,MCBS,Music Concourse/Bandshell,7606,32,1,POINT(-122.468734 37.769431),Weekday,4.256705,NaN,2024-07-01,2025-06-30
15,Golden Gate Park Shuttle,ea33d4691b573336fc9c43c23fa90f65,NaN,NaN,NP-TC-EB,Tennis Center / Dahlia Dell Eastbound,7615,32,1,POINT(-122.458979 37.77123),Weekday,8.429119,NaN,2024-07-01,2025-06-30
5,Golden Gate Park Shuttle,ea33d4691b573336fc9c43c23fa90f65,NaN,NaN,BHLS,Blue Heron Lake,7602,32,1,POINT(-122.476829 37.77084),Weekday,55.567050,NaN,2024-07-01,2025-06-30


In [20]:
cols_to_keep = [
    'feed_key', 'stop_id', 'route_id', 'route_type',
    'n_trips', 'n_routes', 'gtfs_dataset_key'
]

def fuzzy_match_subset(left_df, right_df, threshold=80):
    results = []

    for org in left_df['organization_name'].unique():
        left_group = left_df[left_df['organization_name'] == org]
        right_group = right_df[right_df['organization_name'] == org]

        if right_group.empty:
            continue

        choices = right_group['stop_name'].tolist()

        for idx, row in left_group.iterrows():
            match, score, match_idx = process.extractOne(
                row['stop_name'],
                choices,
                scorer=fuzz.token_set_ratio
            )

            if score >= threshold:
                matched_row = right_group.iloc[match_idx]
                subset = matched_row[cols_to_keep].to_dict()
                matched_name = matched_row['stop_name']
            else:
                subset = {col: None for col in cols_to_keep}
                matched_name = None

            subset.update({
                'left_index': idx,
                'original_stop_name': row['stop_name'],
                'matched_stop_name': matched_name,
                'score': score
            })

            results.append(subset)

    return pd.DataFrame(results)

matches_df = fuzzy_match_subset(
    ridership_subsets['without_route_stopname_match'], 
    stops_aggregated_weekday,
    threshold=80
)

In [21]:
merge_without_route_stopname = (
    ridership_subsets['without_route_stopname_match']
    .merge(
        matches_df,
        left_index=True,
        right_on='left_index',
        how='left',
        indicator=True
    )
    .rename(columns={
        'stop_id_y': 'stop_id',
        'route_id_y': 'route_id'
    })
)

# Keep only rows where the merge matched in both DataFrames
merge_without_route_stopname_matched = merge_without_route_stopname[merge_without_route_stopname['_merge'] == 'both'].copy()

merge_counts = merge_without_route_stopname['_merge'].value_counts()
print(merge_counts)

_merge
both          30
left_only      0
right_only     0
Name: count, dtype: int64


In [22]:
merge_without_route_stopname_matched = standardize_columns(merge_without_route_stopname_matched, master_cols)
merge_without_route_stopname_matched.sample(5)

,organization_name,feed_key,route_id,route_name,stop_id,stop_name,stop_code,n_trips,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,NaN,NaN,70291,Blossom Hill,None,8.0,1.0,None,Weekday,57.459493,NaN,2023-11-01,2025-07-31
16,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,NaN,NaN,70211,Mountain View,None,104.0,3.0,None,Weekday,1906.881882,NaN,2023-11-01,2025-07-31
8,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,NaN,NaN,70251,College Park,None,75.0,1.0,None,Weekday,36.239420,NaN,2023-11-01,2025-07-31
27,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,NaN,NaN,70041,South San Francisco,None,104.0,3.0,None,Weekday,559.501648,NaN,2023-11-01,2025-07-31
1,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,NaN,NaN,70031,Bayshore,None,75.0,1.0,None,Weekday,137.102282,NaN,2023-11-01,2025-07-31


In [23]:
all_ridership_stop_trips_weekday_data = pd.concat([
    merge_with_route_matched,
    merge_without_route_matched,
    merge_with_route_stopcode_matched,
    merge_without_route_stopcode_matched,
    merge_without_route_stopname_matched    
], ignore_index=True)

In [24]:
all_ridership_stop_trips_weekday_data.head(5)

,organization_name,feed_key,route_id,route_name,stop_id,stop_name,stop_code,n_trips,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
0,Gold Coast Transit,3cb676436aad669e52042c0e97a9a051,16,NaN,VNACLR1,.,NaN,32.0,1.0,POINT(-119.294028 34.343645),Weekday,0.000000,3.000000,2025-05-01,2025-05-31
1,Samtrans,db97cc02836aa5f0cf647d80160b23ec,ECR,NaN,345017,1000 El Camino Real-Menlo College,345017,143.0,1.0,POINT(-122.191284 37.457543),Weekday,9.523810,24.571429,2025-08-01,2025-08-31
2,Golden Gate Transit,de77cb40e92fb47fa8d16228977cfb86,130,NaN,40469,1011 Andersen Dr,40469,39.0,1.0,POINT(-122.504252 37.955391),Weekday,1.909091,0.000000,2025-09-01,2025-09-30
3,Samtrans,db97cc02836aa5f0cf647d80160b23ec,61,NaN,343119,1011 Crestview Dr,343119,5.0,1.0,POINT(-122.284103 37.484282),Weekday,1.444444,1.888889,2025-08-01,2025-08-31
4,Samtrans,db97cc02836aa5f0cf647d80160b23ec,46,NaN,340606,1060 Carolan Ave,340606,7.0,1.0,POINT(-122.359627 37.586685),Weekday,10.333333,5.500000,2025-08-01,2025-08-31


In [25]:
all_ridership_stop_trips_weekday_data.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/ridership_trips_routes_weekday.csv", index=False)